# 04 — Breaking the (L_sd, λ) gauge

FF-HEDM spot pixel positions behave, in the small-angle limit, like

$$R_\mathrm{pix} \propto \frac{L_\mathrm{sd}\,\lambda}{d_\mathrm{grain}}$$

where `d_grain` depends on the (refined) grain lattice.  So the
transform `(L_sd, λ) → (k·L_sd, λ/k)` leaves spot pixels approximately
unchanged — **(L_sd, λ) is a near-null mode of the HEDM data Fisher.**

A **powder calibrant** with an *independently known* `d_calibrant`
breaks this: `R_powder ∝ L_sd·λ / d_calibrant_known`, where
`d_calibrant_known` is fixed (Au reference value), not scaled by `k`.

We compute the 2×2 Fisher block on `(Lsd, Wavelength)` at truth under
powder-only / HEDM-only / joint and read off the condition number.
Self-contained synthetic; no LM needed.


In [1]:
import os, math, time
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')   # macOS OpenMP guard
import numpy as np
import torch

import midas_peakfit as mp
from midas_calibrate_v2.forward.panels import PanelLayout
from midas_hkls import Lattice, SpaceGroup
from midas_diffract.hkls import hkls_for_forward_model

# The package's validated synthetic generators (paper fig. 1 source).
import midas_joint_ff_calibrate.runners.run_synthetic_pilatus_joint as R
from midas_joint_ff_calibrate.loss import JointWeights, joint_residual
from midas_joint_ff_calibrate.pipelines.identifiability import fisher_block_rank
from midas_joint_ff_calibrate.pipelines.alternating import AlternatingDriver
from midas_joint_ff_calibrate.pipelines.full_joint import FullJointDriver

# ---- shrink the problem for notebook speed (production paths unchanged)
R.N_GRAINS = 24
R.N_PANELS_Y = 4
R.N_PANELS_Z = 4
R.PANEL_SIZE_Y = 150
R.PANEL_SIZE_Z = 150
R.LSD_UM = 7.0e5            # closer detector -> more panels see Au rings
R.N_RINGS = 6
R.TWO_THETA_MAX_DEG = 14.0
R.N_POWDER_PER_RING = 180

# Loss weights (1/sigma per modality so neither dominates) + gauge lambda.
W_POWDER, W_HEDM, LAMBDA_GAUGE = 1.0e4, 10.0, 1.0e6


def build_problem(seed: int = 2026):
    """Build (layout, truth, grains, ring 2theta/d, powder+HEDM obs)."""
    layout = PanelLayout.regular(R.N_PANELS_Y, R.N_PANELS_Z,
                                 R.PANEL_SIZE_Y, R.PANEL_SIZE_Z,
                                 gap_y=R.GAP_Y, gap_z=R.GAP_Z)
    truth = R.sample_truth(layout, seed=seed)
    grain_eulers, grain_pos, grain_lat = R.sample_truth_grains(seed=seed + 1)

    sg = SpaceGroup.from_number(225)                 # Fm-3m (Au)
    lat = Lattice.for_system('cubic', a=R.AU_LATTICE_A)
    _, thetas, _ = hkls_for_forward_model(
        sg, lat, wavelength_A=R.WAVELENGTH_A,
        two_theta_max_deg=R.TWO_THETA_MAX_DEG, expand_equivalents=False)
    ring_tt, _ = torch.unique(2 * thetas * 180.0 / math.pi,
                              return_inverse=True, sorted=True)
    ring_tt = ring_tt.double()[:R.N_RINGS]
    ring_d = R.WAVELENGTH_A / (2.0 * torch.sin(ring_tt * math.pi / 360.0))

    powder_obs = R.generate_powder_observations(layout, truth, ring_tt, seed=seed + 2)
    hedm_obs = R.generate_hedm_observations(
        layout, truth, grain_eulers, grain_pos, grain_lat, seed=seed + 3)
    return dict(layout=layout, truth=truth,
                grain_eulers=grain_eulers, grain_pos=grain_pos, grain_lat=grain_lat,
                ring_tt=ring_tt, ring_d=ring_d,
                powder_obs=powder_obs, hedm_obs=hedm_obs)


def build_spec(prob, *, refine_grains=False, refine_panels=True):
    """Canonical joint spec: geometry + per-panel deltas + grain blocks."""
    truth = prob['truth']; layout = prob['layout']
    spec = mp.ParameterSpec()
    spec.add(mp.Parameter('Lsd', init=truth.Lsd + 50.0,
                          bounds=(truth.Lsd - 5e3, truth.Lsd + 5e3)))
    spec.add(mp.Parameter('BC_y', init=truth.BC_y + 0.3,
                          bounds=(truth.BC_y - 5.0, truth.BC_y + 5.0)))
    spec.add(mp.Parameter('BC_z', init=truth.BC_z - 0.2,
                          bounds=(truth.BC_z - 5.0, truth.BC_z + 5.0)))
    spec.add(mp.Parameter('ty', init=0.0, refined=False))
    spec.add(mp.Parameter('tz', init=0.0, refined=False))
    spec.add(mp.Parameter('Wavelength', init=R.WAVELENGTH_A, refined=False))
    spec.add(mp.Parameter('pxY', init=R.PX_UM, refined=False))
    spec.add(mp.Parameter('pxZ', init=R.PX_UM, refined=False))
    spec.add(mp.Parameter('RhoD', init=200000.0, refined=False))
    spec.add(mp.Parameter('panel_delta_yz',
                          init=torch.zeros(layout.n_panels(), 2, dtype=torch.float64),
                          bounds=(-3.0, 3.0),
                          prior=mp.GaussianPrior(mean=0.0, std=0.5),
                          refined=refine_panels))
    spec.add(mp.Parameter('panel_delta_theta',
                          init=torch.zeros(layout.n_panels(), dtype=torch.float64),
                          refined=False))
    spec.add(mp.Parameter('grain_euler', init=prob['grain_eulers'],
                          bounds=(-2 * math.pi, 2 * math.pi), refined=refine_grains))
    spec.add(mp.Parameter('grain_pos', init=prob['grain_pos'],
                          bounds=(-1000.0, 1000.0), refined=refine_grains))
    spec.add(mp.Parameter('grain_lattice', init=prob['grain_lat'], refined=False))
    return spec


def make_closures(prob, spec):
    """Return (joint, powder_only, hedm_only) gauge-free residual closures."""
    pf = R.make_powder_residual(prob['powder_obs'], prob['layout'],
                                prob['ring_tt'], ring_d_spacing_A=prob['ring_d'])
    hf = R.make_hedm_residual(prob['hedm_obs'], prob['layout'])

    def joint_fn(u):
        return joint_residual(
            u, powder_residual_fn=pf, hedm_residual_fn=hf, spec=spec,
            weights=JointWeights(w_powder=W_POWDER, w_hedm=W_HEDM,
                                 lambda_gauge=LAMBDA_GAUGE),
            gauge_blocks=[])

    def powder_only(u):   return W_POWDER * pf(u)
    def hedm_only(u):     return W_HEDM * hf(u)
    return joint_fn, powder_only, hedm_only


def seed_unpacked(spec):
    """Dict of every parameter at its init value, as float64 tensors."""
    u = {n: spec.parameters[n].init_tensor() for n in spec.parameters}
    for n in u:
        if not isinstance(u[n], torch.Tensor):
            u[n] = torch.tensor(u[n], dtype=torch.float64)
    return u


def covered_panels(prob):
    p = set(prob['powder_obs']['panel_idx'].tolist())
    h = set(prob['hedm_obs']['panel_idx'].tolist())
    return p, h, (p | h)


DIPlib -- a quantitative image analysis library
Version 3.5.2 (Dec 27 2024)
For more information see https://diplib.org


/Users/hsharma/miniconda3/envs/midas_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
prob = build_problem()
truth = prob['truth']; layout = prob['layout']

# A demo spec with Lsd + Wavelength refined, everything else frozen at truth.
spec = mp.ParameterSpec()
spec.add(mp.Parameter('Lsd', init=truth.Lsd, bounds=(truth.Lsd - 5e3, truth.Lsd + 5e3)))
spec.add(mp.Parameter('BC_y', init=truth.BC_y, refined=False))
spec.add(mp.Parameter('BC_z', init=truth.BC_z, refined=False))
spec.add(mp.Parameter('ty', init=0.0, refined=False))
spec.add(mp.Parameter('tz', init=0.0, refined=False))
spec.add(mp.Parameter('Wavelength', init=R.WAVELENGTH_A,
                      bounds=(R.WAVELENGTH_A * 0.998, R.WAVELENGTH_A * 1.002)))
spec.add(mp.Parameter('pxY', init=R.PX_UM, refined=False))
spec.add(mp.Parameter('pxZ', init=R.PX_UM, refined=False))
spec.add(mp.Parameter('RhoD', init=200000.0, refined=False))
spec.add(mp.Parameter('panel_delta_yz',
                      init=torch.zeros(layout.n_panels(), 2, dtype=torch.float64),
                      refined=False))
spec.add(mp.Parameter('panel_delta_theta',
                      init=torch.zeros(layout.n_panels(), dtype=torch.float64),
                      refined=False))
spec.add(mp.Parameter('grain_euler', init=prob['grain_eulers'],
                      bounds=(-2 * math.pi, 2 * math.pi), refined=False))
spec.add(mp.Parameter('grain_pos', init=prob['grain_pos'],
                      bounds=(-1000.0, 1000.0), refined=False))
spec.add(mp.Parameter('grain_lattice', init=prob['grain_lat'], refined=False))

joint_fn, powder_only, hedm_only = make_closures(prob, spec)
truth_unp = seed_unpacked(spec)
print('refined:', spec.refined_names())


refined: ['Lsd', 'Wavelength']


## The 2×2 Fisher block on (L_sd, λ)

A large eigenvalue ratio (high condition number) means a near-gauge:
the data barely constrains one direction in (L_sd, λ) space.  HEDM-only
should be near-degenerate; powder and joint should break it.


In [3]:
print(f'{"modality":<12s}  {"lambda_min":>12s}  {"lambda_max":>12s}  '
      f'{"cond":>12s}')
for label, fn in [('powder-only', powder_only),
                  ('hedm-only', hedm_only),
                  ('joint', joint_fn)]:
    rep = fisher_block_rank(spec, fn, truth_unp,
                            block_names=['Lsd', 'Wavelength'],
                            sigma_r=1.0, fallback_span=2.0)
    F = rep.fisher.detach()
    eig = torch.linalg.eigvalsh(F).clamp(min=0.0)
    lo, hi = float(eig.min()), float(eig.max())
    cond = hi / max(lo, 1e-300)
    print(f'{label:<12s}  {lo:>12.3e}  {hi:>12.3e}  {cond:>12.3e}')


modality        lambda_min    lambda_max          cond


powder-only      8.624e-03     3.307e+05     3.835e+07
hedm-only        0.000e+00     5.721e+04    5.721e+304
joint            3.609e+03     3.843e+05     1.065e+02


## Reading the result

* **HEDM-only**: the condition number is large — the `(L_sd, λ)`
  direction is a near-null mode, exactly the gauge described above.
  You cannot separately determine `L_sd` and `λ` from HEDM spots alone
  when the grain lattice is also free.
* **Powder-only**: the known calibrant d-spacing anchors the product
  `L_sd·λ` to a fixed ring radius, breaking the gauge (lower
  condition number).
* **Joint**: tightest of all — both modalities contribute.

## Practical recipe

If your experiment refines wavelength alongside geometry, you **must**
include a powder calibrant (with a trusted reference lattice) in the
joint fit — HEDM spots alone leave `(L_sd, λ)` unidentifiable.  This is
the calibration analogue of the multi-distance `(L_sd, a)` story in
`midas_calibrate_v2` notebook 22.
